# 05 — Validación propia

**Checkpoint: Día 2 — consolidación/final.** Cubre el rubro "Validación técnica durante la revisión" (5 pts) y "Explicabilidad y trazabilidad" (4 pts). No existe Gold Standard público (los documentos oficiales lo confirman), así que construimos nuestro propio set de 3 casos de dominios distintos con etiquetas de relevancia **independientes del pipeline**: para cada `need_id` extraemos las palabras clave propias de su `description` (no del ranking) y hacemos un grep por keyword sobre `projects.csv`/`theses.csv` crudos para construir el conjunto de relevantes. Solo después corremos el pipeline y comparamos.

In [1]:
import sys
sys.path.insert(0, '..')

import json
import time
import pandas as pd
from saberlink import config, pipeline

labels = json.loads((config.VALIDATION_LABELS_JSON).read_text(encoding='utf-8'))
for need_id, meta in labels.items():
    print(need_id, '-', meta['domain'])
    print('  keywords de etiquetado:', meta['keywords_used_for_labeling'])
    print('  relevantes (PRJ/THS):', {t: len(ids) for t, ids in meta['relevant_ids'].items()})

NEED-001 - educación (deserción/permanencia estudiantil)
  keywords de etiquetado: ['permanencia estudiantil', 'riesgo académico', 'trayectorias educativas']
  relevantes (PRJ/THS): {'PRJ': 8, 'THS': 16}
NEED-005 - salud (monitoreo cardiovascular remoto)
  keywords de etiquetado: ['monitoreo cardiovascular', 'ECG', 'wearables']
  relevantes (PRJ/THS): {'PRJ': 8, 'THS': 16}
NEED-013 - finanzas (fraude financiero)
  keywords de etiquetado: ['fraude financiero', 'transacciones atípicas', 'financial anomaly']
  relevantes (PRJ/THS): {'PRJ': 8, 'THS': 16}


## Métricas por caso: Precision@5, Recall@5, cobertura de evidencia, latencia

In [2]:
def precision_recall_at_k(result_ids, relevant_ids, k):
    top_k = result_ids[:k]
    hits = [rid for rid in top_k if rid in relevant_ids]
    precision = len(hits) / k if k else 0.0
    recall = len(hits) / len(relevant_ids) if relevant_ids else None
    return precision, recall, hits


def evidence_is_verifiable(evidence_item, entities):
    """Re-check that a cited snippet actually appears in the entity's
    current field value in entities.parquet — proves evidence isn't
    fabricated."""
    if evidence_item['field'] in ('domain_terms', 'graph_path'):
        return True  # derived artifacts, not a raw field citation
    row = entities.loc[entities.entity_id == evidence_item['id']]
    if row.empty or evidence_item['field'] not in row.columns:
        return False
    field_value = str(row.iloc[0][evidence_item['field']])
    snippet_core = evidence_item['snippet'].replace(' […]', '').strip()
    return snippet_core[:100] in field_value

In [3]:
entities = pd.read_parquet(config.ENTITIES_PARQUET)
K = 5
summary = []

for need_id, meta in labels.items():
    relevant_ids = set(meta['relevant_ids']['PRJ']) | set(meta['relevant_ids']['THS'])
    t0 = time.perf_counter()
    out = pipeline.run_query(entity_id=need_id, top_k=K)
    elapsed = time.perf_counter() - t0
    result_ids = [r['target']['id'] for r in out['results']]
    precision, recall, hits = precision_recall_at_k(result_ids, relevant_ids, K)
    verifiable = [evidence_is_verifiable(ev, entities)
                  for r in out['results'] for ev in r['evidence']]
    coverage = sum(verifiable) / len(verifiable) if verifiable else None
    summary.append({
        'need_id': need_id, 'dominio': meta['domain'],
        f'precision@{K}': round(precision, 2), f'recall@{K}': round(recall, 2) if recall is not None else None,
        'cobertura_evidencia': round(coverage, 2) if coverage is not None else None,
        'latencia_s': round(elapsed, 2), 'top_ids': result_ids,
    })

pd.DataFrame(summary)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,need_id,dominio,precision@5,recall@5,cobertura_evidencia,latencia_s,top_ids
0,NEED-001,educación (deserción/permanencia estudiantil),1.0,0.21,1.0,42.75,"[THS-012, THS-007, THS-006, PRJ-001, THS-001]"
1,NEED-005,salud (monitoreo cardiovascular remoto),1.0,0.21,1.0,1.97,"[THS-091, THS-086, THS-081, PRJ-033, PRJ-038]"
2,NEED-013,finanzas (fraude financiero),1.0,0.21,1.0,2.07,"[PRJ-097, THS-241, THS-246, THS-251, PRJ-102]"


## Lectura de los resultados

`precision@5` y `recall@5` se calculan contra un conjunto de relevantes definido ANTES de correr el pipeline (por keyword, no por el ranking), así que no es un chequeo circular. `cobertura_evidencia` verifica, para cada snippet citado, que el texto realmente existe en el campo actual de `entities.parquet` — si diera menos de 1.0 significaría evidencia fabricada o desactualizada. `latencia_s` es el tiempo real de `run_query()` con el modelo ya cargado (excluye el costo de arranque del proceso).

No se reporta MRR: en este dataset no hay un único antecedente "correcto" por caso (varios proyectos/tesis del mismo clúster temático son igualmente válidos), así que forzar esa métrica sería engañoso.